Generar 120 citas: 70% completadas, 30% canceladas de enero 2026 a julio 2026

i.e. citas.csv

In [ ]:
import random
import numpy as np
import pandas as pd

# Fijar semillas para reproducibilidad
random.seed(42)
np.random.seed(42)

# 1. Cargar especialistas desde el CSV existente
df_esp = pd.read_csv('especialistas.csv')

# 2. Cargar pares autorizados especialista-procedimiento
df_auth = pd.read_csv('esp_proc_autorizado.csv')
pairs = df_auth.values.tolist()

clients = list(range(1, 31))
dates = (
    pd.date_range(start='2026-01-01', end='2026-06-30')
    .strftime('%Y-%m-%d')
    .tolist()
)
all_possible_hours = [
    '08:00:00',
    '09:00:00',
    '10:00:00',
    '11:00:00',
    '12:00:00',
    '13:00:00',
    '14:00:00',
    '15:00:00',
    '16:00:00',
    '17:00:00',
]

citas = []
occupied_slots = set()
cita_id = 1

# 3. Generar 120 citas respetando los turnos leídos del CSV
while len(citas) < 120:
  esp, proc = random.choice(pairs)
  cliente = random.choice(clients)
  fecha = random.choice(dates)

  spec_row = df_esp[df_esp['id_especialista'] == esp].iloc[0]
  start_time = spec_row['disponibilidad_inicio']
  end_time = spec_row['disponibilidad_final']

  valid_hours = [h for h in all_possible_hours if start_time <= h <= end_time]
  if not valid_hours:
    continue

  hora = random.choice(valid_hours)

  if (esp, fecha, hora) in occupied_slots or (
      cliente,
      fecha,
      hora,
  ) in occupied_slots:
    continue

  occupied_slots.add((esp, fecha, hora))
  occupied_slots.add((cliente, fecha, hora))

  estado = np.random.choice(['completada', 'cancelada'], p=[0.7, 0.3])

  citas.append({
      'id_cita': cita_id,
      'id_cliente': cliente,
      'id_especialista': esp,
      'id_procedimiento': proc,
      'fecha': fecha,
      'hora': hora,
      'estado': estado,
  })
  cita_id += 1

# 4. Guardar citas.csv
df_citas = pd.DataFrame(citas)
df_citas.to_csv('citas.csv', index=False)

print(
    f'¡Se generaron {len(df_citas)} citas leyendo los horarios de'
    " 'especialistas.csv'!"
)

¡Se generaron 120 citas leyendo los horarios de 'especialistas.csv'!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Generar consumo materiales de la cita

i.e. cita_mov_inventario.csv

In [ ]:
import pandas as pd

# 1. Cargar las tablas necesarias
df_citas = pd.read_csv('citas_enero_junio_2026.csv')
df_mat_req = pd.read_csv('mat_req_proc.csv')

# 2. Filtrar únicamente las citas completadas
citas_completadas = df_citas[df_citas['estado'] == 'completada']

# 3. Generar la relación estricta para cita_mov_inventario
cita_movs = []
mov_id = 1

for _, cita in citas_completadas.iterrows():
    proc_id = cita['id_procedimiento']
    cita_id = cita['id_cita']

    # Obtener los insumos requeridos para este procedimiento
    reqs = df_mat_req[df_mat_req['id_procedimiento'] == proc_id]

    for _, _ in reqs.iterrows():
        cita_movs.append({
            'id_movimiento': mov_id,
            'id_cita': cita_id
        })
        mov_id += 1

# 4. Crear DataFrame y exportar a CSV
df_cita_mov = pd.DataFrame(cita_movs)
df_cita_mov.to_csv('temp_cita_mov_inventario.csv', index=False)

print(f"¡Se generaron {len(df_cita_mov)} registros en 'temp_cita_mov_inventario.csv'!")

¡Se generaron 186 registros en 'temp_cita_mov_inventario.csv'!


Generamos 50 ventas random - variando mas o menos 10% el precio de venta, una vez al mes (sin impuestos) en la tabla de inventario

i.e. temp_producto_venta.csv

In [ ]:
import random
import numpy as np
import pandas as pd

# Fijar semillas para reproducibilidad
random.seed(42)
np.random.seed(42)

# 1. Cargar inventario y filtrar productos de venta general
df_inv = pd.read_csv('inventario.csv')
productos_venta_general = df_inv[df_inv['venta_general'] == 1][
    'id_producto'
].tolist()

# 2. Definir meses y factor de variación mensual de hasta 10% (0.90 a 1.10)
meses_lista = ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']
factores_mes = {mes: random.uniform(0.90, 1.10) for mes in meses_lista}
venta_mes_map = {id_v: random.choice(meses_lista) for id_v in range(1, 51)}

ventas_productos = []

# 3. Generar las exactamente 50 ventas únicas de productos
for id_venta in range(1, 51):
  mes_venta = venta_mes_map[id_venta]
  factor_mensual = factores_mes[mes_venta]

  num_prods = random.randint(1, 2)
  chosen_prods = random.sample(productos_venta_general, num_prods)

  for prod_id in chosen_prods:
    row_prod = df_inv[df_inv['id_producto'] == prod_id]
    precio_base = (
        float(row_prod['precio_base_venta'].values[0])
        if not pd.isna(row_prod['precio_base_venta'].values[0])
        else 20000.0
    )

    # Aplicar variación mensual y calcular cantidad y subtotal
    precio_variado = round(precio_base * factor_mensual, 2)
    cantidad = random.randint(1, 3)
    subtotal = round(cantidad * precio_variado, 2)

    ventas_productos.append({
        'id_venta': id_venta,
        'id_producto': int(prod_id),
        'cantidad': cantidad,
        'subtotal': subtotal,
    })

# 4. Guardar archivo actualizado con la columna cantidad
df_producto_venta = pd.DataFrame(ventas_productos)
df_producto_venta.to_csv('temp_producto_venta.csv', index=False)

print(
    f"Paso completado: {df_producto_venta['id_venta'].nunique()} ventas únicas"
    f" ({len(df_producto_venta)} registros) con 'cantidad' guardados en"
    " 'temp_producto_venta.csv'."
)

Paso completado: 50 ventas únicas (76 registros) con 'cantidad' guardados en 'temp_producto_venta.csv'.


Venta citas variando costo en 10% una vez por mes, basado en las citas generadas

i.e. temp_cita_venta.csv

In [ ]:
import random
import numpy as np
import pandas as pd

# Fijar semillas para reproducibilidad
random.seed(42)
np.random.seed(42)

# 1. Cargar archivos fuente
df_citas = pd.read_csv('citas_enero_junio_2026.csv')
df_proc = pd.read_csv('procedimientos.csv')

# 2. Filtrar únicamente las citas completadas
df_citas_comp = df_citas[df_citas['estado'] == 'completada'].copy()

# 3. Unir con procedimientos para obtener el precio base
df_citas_comp = pd.merge(df_citas_comp, df_proc, on='id_procedimiento', how='left')
df_citas_comp['precio_base'] = df_citas_comp['precio_base'].fillna(80000.0)

# 4. Extraer el mes (YYYY-MM) de la fecha de la cita
df_citas_comp['mes'] = df_citas_comp['fecha'].str[:7]

# 5. Generar UN ÚNICO factor de variación (entre 0.90 y 1.10) por cada mes
meses_lista = df_citas_comp['mes'].unique()
factores_mes = {mes: random.uniform(0.90, 1.10) for mes in meses_lista}
print("Factores de variación por mes:", factores_mes)

# 6. Mapear el factor mensual y calcular el subtotal
df_citas_comp['factor_mes'] = df_citas_comp['mes'].map(factores_mes)
df_citas_comp['subtotal'] = round(df_citas_comp['precio_base'] * df_citas_comp['factor_mes'], 2)

# 7. Seleccionar columnas finales para venta_citas
df_venta_citas = df_citas_comp[['id_cita', 'id_cliente', 'id_procedimiento', 'fecha', 'hora', 'precio_base', 'subtotal']].copy()

# Guardar archivo temporal del paso 1
df_venta_citas.to_csv('temp_cita_venta.csv', index=False)


Factores de variación por mes: {'2026-01': 1.027885359691577, '2026-06': 0.9050021510445334, '2026-02': 0.9550058636738239, '2026-03': 0.9446421476297646, '2026-04': 1.0472942428328025, '2026-05': 1.0353398974845822}


Se unen las tablas de ventas for productos and citas, y se generan las demas columnas requeridas, se actualiza id en child tables

i.e. producto_venta.csv, cita_venta.csv

In [ ]:
import random
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

try:
  df_citas = pd.read_csv('citas.csv', encoding='utf-8-sig')
except FileNotFoundError:
  df_citas = pd.read_csv('citas (1).csv', encoding='utf-8-sig')

df_temp_cita_venta = pd.read_csv('temp_cita_venta.csv', encoding='utf-8-sig')
df_temp_prod_venta = pd.read_csv('temp_producto_venta.csv', encoding='utf-8-sig')
df_clientes = pd.read_csv('clientes.csv', encoding='utf-8-sig')

for df in [df_citas, df_temp_cita_venta, df_temp_prod_venta, df_clientes]:
  df.columns = df.columns.str.strip().str.lower()

lista_clientes = df_clientes['id_cliente'].tolist()

if 'estado' in df_citas.columns:
  df_citas = df_citas[df_citas['estado'].str.strip().str.lower() == 'completada']

cols_temp_cita = [
    col
    for col in ['id_cita', 'id_cliente', 'subtotal']
    if col in df_temp_cita_venta.columns
]
df_citas_enlazadas = pd.merge(
    df_temp_cita_venta[cols_temp_cita],
    df_citas[['id_cita', 'id_cliente', 'fecha', 'hora']],
    on='id_cita',
    how='inner',
    suffixes=('_temp', '_cita'),
)

if 'id_cliente_cita' in df_citas_enlazadas.columns:
  df_citas_enlazadas['id_cliente'] = df_citas_enlazadas['id_cliente_cita']
  df_citas_enlazadas.drop(
      columns=['id_cliente_temp', 'id_cliente_cita'], inplace=True
  )

ids_venta_prod_unicos = df_temp_prod_venta['id_venta'].unique()
mapa_cliente_prod = {
    id_v: random.choice(lista_clientes) for id_v in ids_venta_prod_unicos
}
df_temp_prod_venta['id_cliente'] = df_temp_prod_venta['id_venta'].map(
    mapa_cliente_prod
)

citas_para_mezclar = df_citas_enlazadas.sample(
    n=min(3, len(df_citas_enlazadas)), random_state=42
)
mapa_fusion_citas = {}
for _, row_cita in citas_para_mezclar.iterrows():
  id_prod_elegido = random.choice(ids_venta_prod_unicos)
  mapa_fusion_citas[id_prod_elegido] = row_cita['id_cita']


def get_semestre(fecha_str):
  mes = int(str(fecha_str).split('-')[1])
  return 1 if mes <= 6 else 2


all_txs = []

for _, row in df_citas_enlazadas.iterrows():
  id_cita_orig = row['id_cita']
  prods_asociados = [
      p_id for p_id, c_id in mapa_fusion_citas.items() if c_id == id_cita_orig
  ]
  tipo = 'mixta' if prods_asociados else 'cita'

  subtotal_cita = row['subtotal']
  subtotal_prod = (
      df_temp_prod_venta[
          df_temp_prod_venta['id_venta'].isin(prods_asociados)
      ]['subtotal'].sum()
      if prods_asociados
      else 0.0
  )
  subtotal_venta = subtotal_cita + subtotal_prod
  fecha_h = f"{row['fecha']} {row['hora']}"

  all_txs.append({
      'id_orig': id_cita_orig,
      'prods_orig': prods_asociados,
      'tipo': tipo,
      'id_cliente': row['id_cliente'],
      'fecha_hora': fecha_h,
      'semestre': get_semestre(fecha_h),
      'subtotal': subtotal_venta,
      'is_cita': True,
  })

for pid in ids_venta_prod_unicos:
  if pid not in mapa_fusion_citas.keys():
    subset_prod = df_temp_prod_venta[df_temp_prod_venta['id_venta'] == pid]
    id_cli = subset_prod['id_cliente'].iloc[0]
    subtotal_venta = subset_prod['subtotal'].sum()

    mes_aleatorio = random.choice([
        '2026-01',
        '2026-02',
        '2026-03',
        '2026-04',
        '2026-05',
        '2026-06',
    ])
    fecha_h = (
        f'{mes_aleatorio}-{random.randint(1,28):02d}'
        f' {random.randint(8,18):02d}:00:00'
    )

    all_txs.append({
        'id_orig': pid,
        'prods_orig': [pid],
        'tipo': 'producto',
        'id_cliente': id_cli,
        'fecha_hora': fecha_h,
        'semestre': get_semestre(fecha_h),
        'subtotal': subtotal_venta,
        'is_cita': False,
    })

all_txs.sort(key=lambda x: x['fecha_hora'])

maestro_ventas = []
detalle_cita_venta_rows = []
mapping_prod_orig_a_venta = {}
contador_id_venta = 1
client_semestre_points = {}

for tx in all_txs:
  id_cli = tx['id_cliente']
  semestre = tx['semestre']
  subtotal_venta = tx['subtotal']

  if id_cli not in client_semestre_points:
    client_semestre_points[id_cli] = {}

  available_points = sum([
      pts for sem, pts in client_semestre_points[id_cli].items() if sem <= semestre
  ])
  max_descuento_pesos = min(available_points * 100, subtotal_venta * 0.4)
  max_puntos_redimibles = int(max_descuento_pesos // 100)

  puntos_redimidos = 0
  if max_puntos_redimibles > 0 and random.random() > 0.4:
    puntos_redimidos = random.randint(1, max_puntos_redimibles)
    pts_to_deduct = puntos_redimidos
    for sem in sorted(client_semestre_points[id_cli].keys()):
      if sem <= semestre and pts_to_deduct > 0:
        available_in_sem = client_semestre_points[id_cli][sem]
        deduct = min(available_in_sem, pts_to_deduct)
        client_semestre_points[id_cli][sem] -= deduct
        pts_to_deduct -= deduct

  earned_points = int(subtotal_venta // 1000)
  if semestre not in client_semestre_points[id_cli]:
    client_semestre_points[id_cli][semestre] = 0
  client_semestre_points[id_cli][semestre] += earned_points

  maestro_ventas.append({
      'id_venta': contador_id_venta,
      'id_cliente': id_cli,
      'tipo_venta': tx['tipo'],
      'fecha_hora': tx['fecha_hora'],
      'puntos_redimidos': puntos_redimidos,
  })

  if tx['is_cita']:
    sub_c = df_temp_cita_venta[
        df_temp_cita_venta['id_cita'] == tx['id_orig']
    ]['subtotal'].values[0]
    detalle_cita_venta_rows.append({
        'id_venta': contador_id_venta,
        'id_cita': tx['id_orig'],
        'subtotal': sub_c,
    })

  for p_id in tx['prods_orig']:
    mapping_prod_orig_a_venta[p_id] = contador_id_venta

  contador_id_venta += 1

# Exportar resultados
df_ventas = pd.DataFrame(maestro_ventas)
df_cita_venta_final = pd.DataFrame(detalle_cita_venta_rows)
df_temp_prod_venta['id_venta'] = df_temp_prod_venta['id_venta'].map(
    mapping_prod_orig_a_venta
)

df_ventas.to_csv('ventas.csv', index=False)
df_cita_venta_final.to_csv('cita_venta.csv', index=False)
df_temp_prod_venta[
    ['id_venta', 'id_producto', 'cantidad', 'subtotal']
].dropna().to_csv('producto_venta.csv', index=False)

print(
    '¡Proceso completado! producto_venta.csv incluye id_venta, id_producto,'
    ' cantidad y subtotal.'
)

¡Proceso completado! producto_venta.csv incluye id_venta, id_producto, cantidad y subtotal.


Se calcula el movimiento de inventario asociado a la venta

i.e. venta_mov_inventario

In [ ]:

import pandas as pd

# 1. Cargar la tabla de productos vendidos
df_prod_venta = pd.read_csv('producto_venta.csv')

# 2. Generar la relación directa para venta_mov_inventario
venta_movs = []
mov_id = 1

for _, row in df_prod_venta.iterrows():
  venta_movs.append({'id_movimiento': mov_id, 'id_venta': row['id_venta']})
  mov_id += 1

# 3. Crear DataFrame y exportar a CSV
df_venta_mov = pd.DataFrame(venta_movs)
df_venta_mov.to_csv('temp_venta_mov_inventario.csv', index=False)

print(
    f'¡Se generaron {len(df_venta_mov)} registros en'
    " 'temp_venta_mov_inventario.csv'!"
)

¡Se generaron 76 registros en 'temp_venta_mov_inventario.csv'!


Se unen las tablas de mov inventarios para productos and citas, y se generan las demas columnas requeridas, se actualiza id en child tables

In [ ]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

# 1. Cargar tablas base y transaccionales
citas = pd.read_csv('citas.csv')
ventas = pd.read_csv('ventas.csv')
prod_venta = pd.read_csv('producto_venta.csv')
inventario = pd.read_csv('inventario.csv')

max_inv_id = inventario['id_producto'].max() # Límite superior (32)
all_products = inventario['id_producto'].tolist()

movements = []

# 2. Salidas por Citas (movimiento = 0, cita_venta = 0)
completed_citas = citas[citas['estado'].str.strip().str.lower() == 'completada'].copy()
completed_citas['fecha_hora'] = pd.to_datetime(completed_citas['fecha'] + ' ' + completed_citas['hora'])

for _, row in completed_citas.iterrows():
    movements.append({
        'id_producto': random.choice(all_products),
        'movimiento': 0,
        'cita_venta': 0,
        'fecha_hora': row['fecha_hora'],
        'descripcion': None,
        'cantidad': round(random.uniform(1, 3), 1),
        'source_type': 'cita',
        'source_id': row['id_cita']
    })

# 3. Salidas por Ventas de Productos (movimiento = 0, cita_venta = 1)
prod_venta_full = prod_venta.merge(ventas[['id_venta', 'fecha_hora']], on='id_venta', how='inner')
prod_venta_full['fecha_hora'] = pd.to_datetime(prod_venta_full['fecha_hora'])

for _, row in prod_venta_full.iterrows():
    movements.append({
        'id_producto': row['id_producto'],
        'movimiento': 0,
        'cita_venta': 1,
        'fecha_hora': row['fecha_hora'],
        'descripcion': None,
        'cantidad': round(random.uniform(1, 5), 1),
        'source_type': 'venta',
        'source_id': row['id_venta']
    })

# 4. Entradas por Compras a Proveedores (movimiento = 1, cita_venta = 0)
supplier_phrases = [
    "Due to running low on stock",
    "Restock inventory baseline",
    "Supplier scheduled replenishment",
    "High demand restock",
    "Emergency stock refill"
]

months = ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']
for m in months:
    for _ in range(5):
        day = random.randint(1, 25)
        hour = random.randint(8, 17)
        dt = pd.to_datetime(f"{m}-{day:02d} {hour:02d}:00:00")
        movements.append({
            'id_producto': random.choice(all_products),
            'movimiento': 1,
            'cita_venta': 0,
            'fecha_hora': dt,
            'descripcion': random.choice(supplier_phrases),
            'cantidad': round(random.uniform(10, 50), 1),
            'source_type': 'compra',
            'source_id': None
        })

# 5. Consolidar, ordenar cronológicamente y asignar IDs globales
df_movs = pd.DataFrame(movements)
df_movs['id_producto'] = df_movs['id_producto'].apply(lambda x: x if x <= max_inv_id else ((x - 1) % max_inv_id) + 1)
df_movs = df_movs.sort_values(by='fecha_hora').reset_index(drop=True)
df_movs['id_movimiento'] = range(1, len(df_movs) + 1)

# 6. Exportar tablas finales listas para MySQL
mov_inventario = df_movs[['id_movimiento', 'id_producto', 'movimiento', 'cita_venta', 'fecha_hora', 'descripcion', 'cantidad']]
temp_cita_mov = df_movs[df_movs['source_type'] == 'cita'][['id_movimiento', 'source_id']].rename(columns={'source_id': 'id_cita'})
temp_venta_mov = df_movs[df_movs['source_type'] == 'venta'][['id_movimiento', 'source_id']].rename(columns={'source_id': 'id_venta'})

mov_inventario.to_csv('mov_inventario.csv', index=False)
temp_cita_mov.to_csv('cita_mov_inventario.csv', index=False)
temp_venta_mov.to_csv('venta_mov_inventario.csv', index=False)

print("¡Pipeline de inventario ajustado y exportado exitosamente sin errores de llaves foráneas!")

¡Pipeline de inventario ajustado y exportado exitosamente sin errores de llaves foráneas!


In [ ]:
Se generan 200 visitas, un cliente puede visitar y no comprar ni acceder a procedimiento

In [ ]:
import random
import glob
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

# 1. Cargar tablas necesarias
ventas = pd.read_csv('ventas.csv')
citas = pd.read_csv('citas.csv')
clientes = pd.read_csv('clientes.csv') if 'clientes.csv' in glob.glob('*.csv') else None

if clientes is not None:
  client_ids = clientes['id_cliente'].tolist()
else:
  client_ids = ventas['id_cliente'].unique().tolist()

# 2. Procesar fecha y hora de las ventas existentes
ventas['dt'] = pd.to_datetime(ventas['fecha_hora'])
ventas['fecha'] = ventas['dt'].dt.date
ventas['hora'] = ventas['dt'].dt.strftime('%H:%M:%S')

visitas_ventas = []
for _, row in ventas.iterrows():
  t = str(row['tipo_venta']).strip().lower()
  c_flag = 1 if t in ['cita', 'mixto'] else (
      1 if t == 'cita' else 0
  )  # Manejo de flags
  # Si es mixto, ambos flags son 1
  c_flag = 1 if t in ['cita', 'mixto'] else 0
  p_flag = 1 if t in ['producto', 'mixto'] else 0

  visitas_ventas.append({
      'id_cliente': row['id_cliente'],
      'fecha': str(row['fecha']),
      'hora': str(row['hora']),
      'cita': c_flag,
      'compra': p_flag,
      'dt': row['dt'],
  })

# 3. Rellenar hasta alcanzar 200 visitas con registros donde no hubo cita ni compra (0, 0)
current_count = len(visitas_ventas)
target_count = 200
extra_needed = max(0, target_count - current_count)

start_date = pd.to_datetime('2026-01-01')
end_date = pd.to_datetime('2026-06-30')

extra_visitas = []
for _ in range(extra_needed):
  random_days = random.randint(0, (end_date - start_date).days)
  random_dt = start_date + pd.Timedelta(
      days=random_days,
      hours=random.randint(8, 17),
      minutes=random.randint(0, 59),
  )
  cid = random.choice(client_ids)
  extra_visitas.append({
      'id_cliente': cid,
      'fecha': random_dt.strftime('%Y-%m-%d'),
      'hora': random_dt.strftime('%H:%M:%S'),
      'cita': 0,
      'compra': 0,
      'dt': random_dt,
  })

# 4. Consolidar, ordenar cronológicamente y asignar reg_visita incremental
all_visitas = visitas_ventas + extra_visitas
df_visitas = pd.DataFrame(all_visitas)
df_visitas = df_visitas.sort_values(by='dt').reset_index(drop=True)
df_visitas['reg_visita'] = range(1, len(df_visitas) + 1)

# 5. Seleccionar columnas finales y exportar a CSV
df_visita_final = df_visitas[
    ['reg_visita', 'id_cliente', 'fecha', 'hora', 'cita', 'compra']
]
df_visita_final.to_csv('visita.csv', index=False)

print(
    f'¡Se generaron {len(df_visita_final)} registros de visitas en orden'
    " cronológico ('visita.csv')!"
)

¡Se generaron 200 registros de visitas en orden cronológico ('visita.csv')!


Fix para cita venta, ya que codigo original desfaso precios

In [ ]:
import numpy as np
import pandas as pd

# 1. Cargar los archivos CSV existentes
citas = pd.read_csv('citas.csv')
cita_venta = pd.read_csv('cita_venta.csv')
procedimientos = pd.read_csv('procedimientos.csv')

# 2. Unir las tablas para relacionar cada venta con su procedimiento y precio base
merged = cita_venta.merge(
    citas[['id_cita', 'id_procedimiento']], on='id_cita', how='left'
)
merged = merged.merge(
    procedimientos[['id_procedimiento', 'precio_base']],
    on='id_procedimiento',
    how='left',
)

# 3. Aplicar una variación aleatoria controlada entre el -10% y el +10% (factores de 0.90 a 1.10)
np.random.seed(42)
variation = np.random.uniform(0.90, 1.10, size=len(merged))
merged['subtotal'] = (merged['precio_base'] * variation).round(2)

# 4. Seleccionar las columnas originales y sobrescribir el archivo CSV
new_cita_venta = merged[['id_venta', 'id_cita', 'subtotal']]
new_cita_venta.to_csv('cita_venta-2.csv', index=False)

print('¡Archivo cita_venta.csv actualizado con éxito!')

¡Archivo cita_venta.csv actualizado con éxito!
